# Transformer-Based Language Model Benchmarking

Integration of pre-trained transformer models for text representation:
- **Sentence-BERT** (all-MiniLM-L6-v2) for sentence embeddings
- **BERT** (bert-base-uncased) for contextual embeddings
- Comparison with TF-IDF baseline
- Downstream classification with transformer features

## Section 1: Setup

In [ ]:
import pandas as pd
import numpy as np
import re, warnings, time, os
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report)
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier
import joblib

try:
    from sentence_transformers import SentenceTransformer
    HAS_SBERT = True
    print('sentence-transformers available.')
except ImportError:
    HAS_SBERT = False
    print('sentence-transformers not installed. pip install sentence-transformers')

try:
    from transformers import AutoTokenizer, AutoModel
    import torch
    HAS_TRANSFORMERS = True
    print('transformers (HuggingFace) available.')
except ImportError:
    HAS_TRANSFORMERS = False
    print('transformers not installed. pip install transformers torch')

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
print('Setup complete.')

In [ ]:
# Load and preprocess data
df = pd.read_csv('../data/cumulative_ai_customer_communication_dataset.csv', low_memory=False)
df['target'] = (df['csat_score'] >= 4).astype(int)

# Keep original messages for transformers (they handle their own tokenization)
df['raw_message'] = df['customer_message'].fillna('').astype(str)

# Also create cleaned version for TF-IDF baseline
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
def preprocess_text(text):
    if pd.isna(text) or not isinstance(text, str): return ''
    text = text.lower().encode('ascii','ignore').decode('ascii')
    text = re.sub(r'[^a-z\s]','',text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t)>1]
    return ' '.join(tokens)

df['cleaned_message'] = df['customer_message'].apply(preprocess_text)
print(f'Dataset: {df.shape[0]} rows, Positive rate: {df["target"].mean()*100:.1f}%')

# Train-test split (use same indices for all representations)
train_idx, test_idx = train_test_split(df.index, test_size=0.2, random_state=42, stratify=df['target'])
y_train = df.loc[train_idx, 'target']
y_test = df.loc[test_idx, 'target']
print(f'Train: {len(train_idx)}, Test: {len(test_idx)}')

## Section 2: TF-IDF Baseline

In [ ]:
# TF-IDF baseline
print('=== TF-IDF Baseline ===')
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=5)
X_tfidf_train = tfidf.fit_transform(df.loc[train_idx, 'cleaned_message'])
X_tfidf_test = tfidf.transform(df.loc[test_idx, 'cleaned_message'])

lr_tfidf = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr_tfidf.fit(X_tfidf_train, y_train)
y_pred = lr_tfidf.predict(X_tfidf_test)
y_prob = lr_tfidf.predict_proba(X_tfidf_test)[:,1]
print(f'  LR+TF-IDF: Acc={accuracy_score(y_test,y_pred)*100:.2f}%, F1={f1_score(y_test,y_pred)*100:.2f}%, AUC={roc_auc_score(y_test,y_prob):.4f}')

baseline_results = {'Accuracy': accuracy_score(y_test,y_pred), 'F1': f1_score(y_test,y_pred), 'AUC': roc_auc_score(y_test,y_prob)}

## Section 3: Sentence-BERT Embeddings

In [ ]:
# Sentence-BERT embeddings
if HAS_SBERT:
    print('=== Sentence-BERT (all-MiniLM-L6-v2) ===')
    sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

    print('Encoding training texts...')
    start = time.time()
    X_sbert_train = sbert_model.encode(
        df.loc[train_idx, 'raw_message'].tolist(),
        batch_size=64, show_progress_bar=True, normalize_embeddings=True
    )
    print(f'  Train embeddings: {X_sbert_train.shape} ({time.time()-start:.1f}s)')

    print('Encoding test texts...')
    X_sbert_test = sbert_model.encode(
        df.loc[test_idx, 'raw_message'].tolist(),
        batch_size=64, show_progress_bar=True, normalize_embeddings=True
    )
    print(f'  Test embeddings: {X_sbert_test.shape}')

    # Classify with LR
    lr_sbert = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
    lr_sbert.fit(X_sbert_train, y_train)
    y_pred_sbert = lr_sbert.predict(X_sbert_test)
    y_prob_sbert = lr_sbert.predict_proba(X_sbert_test)[:,1]
    print(f'  LR+SBERT: Acc={accuracy_score(y_test,y_pred_sbert)*100:.2f}%, F1={f1_score(y_test,y_pred_sbert)*100:.2f}%, AUC={roc_auc_score(y_test,y_prob_sbert):.4f}')

    # XGBoost with SBERT
    neg_c=(y_train==0).sum(); pos_c=(y_train==1).sum()
    xgb_sbert = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
        scale_pos_weight=neg_c/pos_c, random_state=42, eval_metric='logloss', use_label_encoder=False)
    xgb_sbert.fit(X_sbert_train, y_train)
    y_pred_xgb_s = xgb_sbert.predict(X_sbert_test)
    y_prob_xgb_s = xgb_sbert.predict_proba(X_sbert_test)[:,1]
    print(f'  XGB+SBERT: Acc={accuracy_score(y_test,y_pred_xgb_s)*100:.2f}%, F1={f1_score(y_test,y_pred_xgb_s)*100:.2f}%, AUC={roc_auc_score(y_test,y_prob_xgb_s):.4f}')

    sbert_results = {
        'LR+SBERT': {'Acc':accuracy_score(y_test,y_pred_sbert),'F1':f1_score(y_test,y_pred_sbert),'AUC':roc_auc_score(y_test,y_prob_sbert)},
        'XGB+SBERT': {'Acc':accuracy_score(y_test,y_pred_xgb_s),'F1':f1_score(y_test,y_pred_xgb_s),'AUC':roc_auc_score(y_test,y_prob_xgb_s)}
    }
else:
    print('Skipping SBERT (not installed)')
    sbert_results = {}

## Section 4: BERT Embeddings (CLS Token)

In [ ]:
# BERT CLS token embeddings
if HAS_TRANSFORMERS:
    print('=== BERT (bert-base-uncased) CLS Embeddings ===')
    tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
    bert_model = AutoModel.from_pretrained('bert-base-uncased')
    bert_model.eval()

    def get_bert_embeddings(texts, batch_size=32):
        embeddings = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            encoded = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors='pt')
            with torch.no_grad():
                outputs = bert_model(**encoded)
            cls_emb = outputs.last_hidden_state[:, 0, :].numpy()
            embeddings.append(cls_emb)
            if (i//batch_size) % 20 == 0:
                print(f'    Batch {i//batch_size}/{len(texts)//batch_size}')
        return np.vstack(embeddings)

    # Use subset for BERT (full dataset may be slow)
    MAX_SAMPLES = min(10000, len(train_idx))
    sample_train_idx = np.random.RandomState(42).choice(train_idx, MAX_SAMPLES, replace=False)
    sample_test_idx = test_idx[:min(2000, len(test_idx))]
    y_train_sample = df.loc[sample_train_idx, 'target']
    y_test_sample = df.loc[sample_test_idx, 'target']

    print(f'  Using {MAX_SAMPLES} train, {len(sample_test_idx)} test samples for BERT')
    start = time.time()
    X_bert_train = get_bert_embeddings(df.loc[sample_train_idx, 'raw_message'].tolist())
    X_bert_test = get_bert_embeddings(df.loc[sample_test_idx, 'raw_message'].tolist())
    print(f'  BERT embeddings: {X_bert_train.shape}, Time: {time.time()-start:.1f}s')

    # Classify
    lr_bert = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
    lr_bert.fit(X_bert_train, y_train_sample)
    y_pred_bert = lr_bert.predict(X_bert_test)
    y_prob_bert = lr_bert.predict_proba(X_bert_test)[:,1]
    print(f'  LR+BERT: Acc={accuracy_score(y_test_sample,y_pred_bert)*100:.2f}%, F1={f1_score(y_test_sample,y_pred_bert)*100:.2f}%, AUC={roc_auc_score(y_test_sample,y_prob_bert):.4f}')
    bert_results = {'Acc':accuracy_score(y_test_sample,y_pred_bert),'F1':f1_score(y_test_sample,y_pred_bert),'AUC':roc_auc_score(y_test_sample,y_prob_bert)}
else:
    print('Skipping BERT (transformers not installed)')
    bert_results = {}

## Section 5: Comparison Summary

In [ ]:
# Summary comparison
print('=== TRANSFORMER BENCHMARKING RESULTS ===\n')
comparison = [{'Model':'TF-IDF + LR (Baseline)','Accuracy':baseline_results['Accuracy'],
    'F1':baseline_results['F1'],'AUC':baseline_results['AUC'],'Embedding Dim':X_tfidf_train.shape[1]}]

if HAS_SBERT:
    comparison.append({'Model':'SBERT + LR','Accuracy':sbert_results['LR+SBERT']['Acc'],
        'F1':sbert_results['LR+SBERT']['F1'],'AUC':sbert_results['LR+SBERT']['AUC'],'Embedding Dim':X_sbert_train.shape[1]})
    comparison.append({'Model':'SBERT + XGBoost','Accuracy':sbert_results['XGB+SBERT']['Acc'],
        'F1':sbert_results['XGB+SBERT']['F1'],'AUC':sbert_results['XGB+SBERT']['AUC'],'Embedding Dim':X_sbert_train.shape[1]})
if HAS_TRANSFORMERS and bert_results:
    comparison.append({'Model':'BERT-CLS + LR','Accuracy':bert_results['Acc'],
        'F1':bert_results['F1'],'AUC':bert_results['AUC'],'Embedding Dim':768})

comp_df = pd.DataFrame(comparison)
disp = comp_df.copy()
for c in ['Accuracy','F1']: disp[c]=(disp[c]*100).round(2).astype(str)+'%'
disp['AUC']=disp['AUC'].round(4)
print(disp.to_string(index=False))

# Visualization
fig,ax=plt.subplots(figsize=(10,5))
x=np.arange(len(comp_df));w=0.25
ax.bar(x-w,comp_df['Accuracy'],w,label='Accuracy',color='steelblue')
ax.bar(x,comp_df['F1'],w,label='F1',color='coral')
ax.bar(x+w,comp_df['AUC'],w,label='AUC',color='mediumseagreen')
ax.set_xticks(x);ax.set_xticklabels(comp_df['Model'],rotation=15,ha='right')
ax.set_title('Transformer vs Traditional: Performance Comparison');ax.set_ylabel('Score');ax.set_ylim(0,1.1)
ax.legend();plt.tight_layout()
plt.savefig('../models/transformer_comparison.png',dpi=150,bbox_inches='tight');plt.show()

# Save embeddings for reuse
if HAS_SBERT:
    np.save('../models/sbert_train_embeddings.npy', X_sbert_train)
    np.save('../models/sbert_test_embeddings.npy', X_sbert_test)
    print('SBERT embeddings saved.')
comp_df.to_csv('../models/transformer_results.csv',index=False)
print('Results saved.')